# AR6 WGII adaptation feasibility — extraction

Extracts the multidimensional feasibility ratings for adaptation options from **AR6 WGII Chapter 18 Supplementary Material, Cross-Chapter Box FEASIB** (Tables SMCCB FEASIB.1–7) into one tidy per-cell table. Indicator ratings are encoded as colored bars in the PDF, so this notebook reads cell-fill colors with `pdfplumber` — the same technique the SR1.5 mitigation review used.

- Source (indicator detail): `IPCC_AR6_WGII_Chapter18_SM.pdf`, FEASIB section pp. 18SM-11 to 18SM-31. IPCC copyright — file lives in `sample/` (gitignored), not committed. Re-download URL in the review README.
- Source (figure roll-up): DDC record 5916, doi:10.48490/tnk5-rv35 (CC BY 4.0).
- Review README: `../../README.md`. Counterpart: `reviews/ipcc/ipcc-sr15-mitigation-feasibility`.

**Restart-and-run-all requires the PDF in `sample/`.** Output: `data/ar6_wg2_feasibility_per_cell.csv` (460 rows) and `data/ar6_wg2_feasibility_per_option.csv` (23 options).

In [ ]:
from pathlib import Path
import pdfplumber, pandas as pd

RELEASE = Path.cwd()
SM_PDF = RELEASE / "sample" / "IPCC_AR6_WGII_Chapter18_SM.pdf"
OUT_CELL = RELEASE / "data" / "ar6_wg2_feasibility_per_cell.csv"
OUT_OPTION = RELEASE / "data" / "ar6_wg2_feasibility_per_option.csv"
if not SM_PDF.exists():
    raise FileNotFoundError(
        f"Source PDF not found at {SM_PDF}.\nDownload (IPCC copyright; keep in sample/, do not commit):\n"
        "  https://www.ipcc.ch/report/ar6/wg2/downloads/report/IPCC_AR6_WGII_Chapter18_SM.pdf")
pdf = pdfplumber.open(SM_PDF)
print(f"opened {len(pdf.pages)} pages")

## Reference constants (all verified against the source)

Six dimensions × their indicators (the taxonomy, read from the SM tables); the seven feasibility tables and their option columns (positional — table 3 has **three** columns because *Forest-based adaptation* merges sustainable-forest-management with reforestation); the table-caption y-positions that bound each table (tables split mid-page, so we switch option sets at caption boundaries, not page boundaries); and the **bar color palette** (dark blue = High, mid blue = Medium, pale blue = Low), calibrated from the legend and confirmed against the distinct fills on the FEASIB pages.

In [ ]:
TAXONOMY = {
    "Economic": ["Microeconomic viability", "Macroeconomic viability", "Socioeconomic vulnerability reduction potential", "Employment and productivity enhancement potential"],
    "Technological": ["Technical resource availability", "Risks mitigation potential"],
    "Institutional": ["Political acceptability", "Legal and regulatory acceptability", "Institutional capacity and administrative feasibility", "Transparency and accountability potential"],
    "Socio-cultural": ["Social co-benefits (health, education)", "Socio-cultural acceptability", "Social and regional inclusiveness", "Gender equity", "Intergenerational equity"],
    "Environmental/ecological": ["Ecological capacity", "Adaptive capacity/resilience"],
    "Geophysical": ["Physical feasibility", "Land use change enhancement potential", "Hazard risk reduction potential"],
}
IND2DIM = {i: d for d, ind in TAXONOMY.items() for i in ind}
# (label key, indicator) — keys are short prefixes so they survive line-break hyphenation (e.g. 'Intergenera-')
IND_KEY = [("Microeco","Microeconomic viability"),("Macroeco","Macroeconomic viability"),("Socioeco","Socioeconomic vulnerability reduction potential"),("Employ","Employment and productivity enhancement potential"),("Technical","Technical resource availability"),("Risks","Risks mitigation potential"),("Political","Political acceptability"),("Legal","Legal and regulatory acceptability"),("Institut","Institutional capacity and administrative feasibility"),("Transpar","Transparency and accountability potential"),("co-benefit","Social co-benefits (health, education)"),("Socio-cultural","Socio-cultural acceptability"),("inclus","Social and regional inclusiveness"),("Gender","Gender equity"),("Intergenera","Intergenerational equity"),("Ecolog","Ecological capacity"),("Adapt","Adaptive capacity/resilience"),("Physical","Physical feasibility"),("Land","Land use change enhancement potential"),("Hazard","Hazard risk reduction potential")]
TABLES = {
    1: ["Resilient power systems", "Energy reliability", "Water use efficiency"],
    2: ["Integrated coastal zone management", "Sustainable aquaculture and fisheries", "Coastal defence and hardening"],
    3: ["Forest-based adaptation", "Biodiversity management and ecosystem connectivity", "Agro-forestry"],
    4: ["Improved cropland management", "Efficient livestock systems", "Livelihood diversification", "Water use efficiency and water resource management"],
    5: ["Sustainable land use and urban planning", "Green infrastructure and ecosystem services", "Sustainable water management"],
    6: ["Disaster risk management", "Climate services including EWS", "Risk spreading and sharing"],
    7: ["Population health and health systems", "Social safety nets", "Planned relocation and resettlement", "Human migration and displacement"],
}
TRANS = {1:"Energy system transition",2:"Land and ecosystem transition",3:"Land and ecosystem transition",4:"Land and ecosystem transition",5:"Urban and infrastructure transition",6:"Overarching",7:"Cross-sectoral"}
# (table, page_index_0based, y) of each 'Table SMCCB FEASIB.N |' caption — verified
CAPTIONS = [(1,10,249),(2,12,513),(3,16,386),(4,19,51),(5,23,405),(6,25,93),(7,27,51)]
WHITE, CREAM = (1.0,1.0,1.0), (1.0,0.954,0.852)
print(f"{sum(len(v) for v in TABLES.values())} options across {len(TABLES)} tables; {len(IND_KEY)} indicators")

## Clean — geometry helpers

Bars are narrow (~20pt) colored rects; option columns come from the cream header cells (with a bar-cluster fallback); indicator rows from the label column (x 68–left, which excludes the dimension labels at x~48–60). `table_at` resolves which table a (page, y) belongs to via the caption boundaries, so a page holding the tail of one table and the head of the next is split correctly.

In [ ]:
def rgb(r):
    c = r.get("non_stroking_color")
    return tuple(round(x, 3) for x in c) if isinstance(c, (list, tuple)) else c

def band(v):
    R = v[0]
    return "High" if R < 0.35 else ("Medium" if R < 0.70 else "Low")  # dark / mid / pale blue

def bars_of(pg):
    return [r for r in pg.rects if 15 < float(r['x1'])-float(r['x0']) < 26 and rgb(r) not in (WHITE, CREAM)]

def table_at(page, y):
    best = None
    for n, cp, cy in CAPTIONS:
        if (cp < page) or (cp == page and cy <= y+2): best = n
    return best

def header_rows(pg):
    cream = [r for r in pg.rects if rgb(r) == CREAM and float(r['x1'])-float(r['x0']) > 40 and float(r['x0']) > 140]
    groups = {}
    for r in cream: groups.setdefault(round(float(r['top'])/6)*6, []).append(r)
    out = []
    for _, cells in groups.items():
        if len(cells) < 2: continue
        xs = sorted((float(r['x0']), float(r['x1'])) for r in cells)
        out.append((min(float(r['top']) for r in cells), xs))
    return sorted(out)

HEADERS = [(pi, yt, xs) for pi in range(10, 32) for yt, xs in header_rows(pdf.pages[pi])]

def gov_cols(page, y):
    best = None
    for hp, hy, xs in HEADERS:
        if (hp < page) or (hp == page and hy <= y+2):
            if best is None or (hp, hy) > (best[0], best[1]): best = (hp, hy, xs)
    return best[2] if best else None

def indicator_rows(pg, left):
    labs = [w for w in pg.extract_words() if 68 <= float(w['x0']) < left]
    out = []
    for key, ind in IND_KEY:
        prim = [w for w in labs if w['text'].startswith(key)]
        if prim:
            yc = sum(float(w['top'])+float(w['bottom']) for w in prim) / (2*len(prim))
            out.append((ind, yc))
    return out
print(f"detected {len(HEADERS)} header rows across the FEASIB pages")

## Clean — per-cell extraction

For each FEASIB page, build the (indicator-row × option-column) grid, then assign each colored bar and each `NE`/`LE`/`NA` text flag to its nearest row + containing column. A bar wins over a flag in the same cell. Rows on page 31 below the synergy caption (y≥0.4k) are skipped — they belong to Tables 8–9 (synergies), a different structure.

In [ ]:
def cols_for(pi, yc, bars):
    n = table_at(pi, yc)
    if n is None: return None, None
    opts = TABLES[n]; xs = gov_cols(pi, yc)
    if not xs or len(xs) != len(opts):  # fallback: cluster this table's bars on the page
        seg = [r for r in bars if table_at(pi, (float(r['top'])+float(r['bottom']))/2) == n]
        cs = sorted(set(round(float(r['x0'])) for r in seg)); cl = []
        for x in cs:
            if cl and x-cl[-1][-1] < 10: cl[-1].append(x)
            else: cl.append([x])
        cen = [sum(c)/len(c) for c in cl]
        if len(cen) != len(opts): return None, None
        xs = [(c-3, c+115) for c in cen]
    return n, list(zip(xs, opts))

recs = []
for pi in range(10, 32):
    pg = pdf.pages[pi]; bars = bars_of(pg)
    if not bars: continue
    left = min(float(r['x0']) for r in bars)
    flags = [w for w in pg.extract_words() if w['text'] in ("NE", "LE", "NA")]
    page_rows = [(ind, yc) for ind, yc in indicator_rows(pg, left) if not (pi == 31 and yc >= 400)]
    if not page_rows: continue
    nearest = lambda y: min(page_rows, key=lambda r: abs(r[1]-y))
    cell = {}
    for ind, yc in page_rows:
        n, co = cols_for(pi, yc, bars)
        if not co: continue
        for (x0, x1), opt in co:
            cell[(ind, opt)] = dict(transition=TRANS[n], table=n, option=opt, dimension=IND2DIM[ind], indicator=ind, rating=None, flag=None, page=pi+1)
    for r in bars:  # bars -> nearest row + containing column
        ind, yc = nearest((float(r['top'])+float(r['bottom']))/2); n, co = cols_for(pi, yc, bars)
        if not co: continue
        for (x0, x1), opt in co:
            if x0-2 <= float(r['x0']) < x1 and cell.get((ind, opt), {}).get("rating") is None and (ind, opt) in cell:
                cell[(ind, opt)]["rating"] = band(rgb(r))
    for f in flags:  # flags -> nearest row + containing column (only where no bar)
        ind, yc = nearest((float(f['top'])+float(f['bottom']))/2); n, co = cols_for(pi, yc, bars)
        if not co: continue
        for (x0, x1), opt in co:
            c = cell.get((ind, opt))
            if c and x0-2 <= float(f['x0']) < x1 and c["rating"] is None and c["flag"] is None:
                c["flag"] = f['text']
    recs.extend(cell.values())

cells = pd.DataFrame(recs).drop_duplicates(["option", "indicator"]).sort_values(["table", "option", "dimension", "indicator"]).reset_index(drop=True)
cells.head(8)

## Validate

Assertions, not eyeballing. Counts, taxonomy closure, palette closure, mutual exclusivity of rating/flag, and three spot-checks read off the rendered PDF pages (energy p10, table 2 + table 3 on p16). If any fail, do not export.

In [ ]:
n_opt = sum(len(v) for v in TABLES.values())
assert cells.option.nunique() == n_opt == 23, cells.option.nunique()
assert len(cells) == n_opt * 20 == 460, len(cells)
assert cells.groupby("option").indicator.nunique().eq(20).all(), "every option must score all 20 indicators"
assert set(cells.rating.dropna()) <= {"High", "Medium", "Low"}
assert set(cells.flag.dropna()) <= {"NE", "LE", "NA"}
assert cells.dropna(subset=["rating", "flag"]).empty, "a cell is rated or flagged, never both"
assert (cells.dimension == cells.indicator.map(IND2DIM)).all()

def cellval(opt, ind):
    r = cells[(cells.option == opt) & (cells.indicator == ind)].iloc[0]
    return r.rating if pd.notna(r.rating) else r.flag
# spot-checks vs rendered pages
assert cellval("Resilient power systems", "Microeconomic viability") == "High"        # p10, dark bar
assert cellval("Water use efficiency", "Employment and productivity enhancement potential") == "NE"  # p10, 'NE'
assert cellval("Forest-based adaptation", "Microeconomic viability") == "Low"          # p16, pale bar
assert cellval("Biodiversity management and ecosystem connectivity", "Macroeconomic viability") == "LE"  # p16, 'LE'
assert cellval("Coastal defence and hardening", "Land use change enhancement potential") == "LE"        # p16, 'LE'
print("all assertions passed:", len(cells), "cells,", cells.rating.notna().sum(), "ratings,", cells.flag.notna().sum(), "flags")

## Export

One tidy per-cell table (committed) plus a per-option × dimension roll-up that mirrors the figure-level grain (SPM.4 / DDC 5916). **The `flag` column contains the literal string `NA`** — read it back with `keep_default_na=False` or `na_values=[]`, otherwise pandas turns `NA` into `NaN` and you silently lose the 22 not-applicable cells.

In [ ]:
OUT_CELL.parent.mkdir(parents=True, exist_ok=True)
cells.to_csv(OUT_CELL, index=False)
dom = lambda s: (s.dropna().mode().iat[0] if s.notna().any() else None)
dimorder = ["Economic", "Technological", "Institutional", "Socio-cultural", "Environmental/ecological", "Geophysical"]
roll = (cells.groupby(["transition", "table", "option", "dimension"])["rating"].agg(dom).unstack("dimension").reindex(columns=dimorder))
roll.to_csv(OUT_OPTION)
# round-trip check that 'NA' survived
back = pd.read_csv(OUT_CELL, keep_default_na=False)
assert (back.flag == "NA").sum() == 22, "NA flags lost on round-trip"
print(f"wrote {OUT_CELL.name} ({len(cells)}) and {OUT_OPTION.name} ({len(roll)})")

## Visualise & fit

Inline only. Two cells the review cites: the rating mix per dimension (where the data discriminates options vs. where it is mostly insufficient-evidence), and the share of cells that are NE/LE/NA per dimension — the analogue of the SR1.5 'missing evidence pulls feasibility down' caveat. Geophysical is mostly NA; Institutional and Socio-cultural carry most of the NE/LE.

In [ ]:
import matplotlib.pyplot as plt
state = cells.rating.fillna(cells.flag)
mix = (cells.assign(state=state).groupby(["dimension", "state"]).size().unstack("state").reindex(dimorder).fillna(0)[["High", "Medium", "Low", "NE", "LE", "NA"]])
ax = mix.plot(kind="barh", stacked=True, figsize=(9, 4), color={"High":"#00528a","Medium":"#81a1c9","Low":"#dbe6f3","NE":"#bbbbbb","LE":"#dddddd","NA":"#f0e8de"})
ax.set_title("AR6 WGII adaptation feasibility — cell states by dimension (23 options)")
ax.set_xlabel("cells"); ax.set_ylabel(""); ax.invert_yaxis(); plt.tight_layout(); plt.show()
mix.astype(int)

## Findings

- **460 cells extracted, zero blank**: 390 ratings (131 High, 191 Medium, 68 Low) + 70 flags (27 LE, 21 NE, 22 NA). Every one of the 23 options scores all 20 indicators.
- **Color palette confirmed**: bars come in exactly three blue families — dark `R<0.35` (High), mid `R≈0.5` (Medium), pale `R≈0.85` (Low); cream `(1.0,0.954,0.852)` is header/label shading, never a rating.
- **Table 3 has three columns, not four**: *Forest-based adaptation* merges sustainable-forest-management/conservation with reforestation/afforestation — matching how SPM.4 labels the option. The caption lists four phrases but the table renders three columns. (Parsing note in README.)
- **Tables straddle page breaks**: e.g. the energy table's Environmental/Geophysical rows sit at the top of the page that also starts table 2. Option sets switch at caption y-positions, not page boundaries.
- **Hyphenation**: indicator labels line-break mid-word (`Intergenera-`/`tional`), so label matching uses short prefixes.
- **NA ≠ NaN**: the literal string `NA` (22 not-applicable cells, mostly Geophysical) is destroyed by pandas' default NA parsing on read — a real trap for downstream use, asserted on round-trip above and flagged in the README.
- **Coverage shape**: Geophysical is dominated by NA (physical-feasibility / land-use indicators are often not applicable to non-land options); Institutional and Socio-cultural carry most of the NE/LE. Economic, Technological and Socio-cultural are the most fully-rated and therefore the most discriminating dimensions.